# NeuroMind — generate the real model bundle (Google Colab)

This notebook trains the binary speech encoder and multimodal Transformer from **your real, patient-aligned datasets**, validates the artifacts, installs them into the prepared project, and creates `neuromind-model-bundle.zip`.

**Important:** the prepared project ZIP contains source code, not pre-trained weights. This notebook creates weights by running training. It cannot run correctly if the datasets are missing, synthetic, or have different patient IDs/labels across modalities. This is a research integration workflow, **not clinical validation**. Do not use results to diagnose people or make clinical decisions.

## Before running
1. Download `NeuroMind_PRODUCTION_PREPARED_v15.zip` and upload that ZIP to the top level of Google Drive (`My Drive/NeuroMind_PRODUCTION_PREPARED_v15.zip`).
2. Keep your data under a Google Drive folder named `NeuroMind` (a trailing space in the folder name is tolerated) with `01_Datasets` inside it.
3. Speech recordings must be arranged as `Speech Dataset/<label>/<patient_id>/<audio-file>` where labels are the Non-Dementia/Healthy and Dementia groups.
4. The CSV files must be named `Memory_Dataset.csv`, `Attention_Dataset.csv`, and `Lifestyle_Dataset.csv`. Patient IDs and labels must match the speech records.
5. In Colab, choose **Runtime → Change runtime type → T4 GPU** (or another available GPU), then run cells from top to bottom.

The notebook saves intermediate data and the finished model bundle to your Google Drive. Training can take a while; do not interrupt a cell while it is writing model files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys

DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_ZIP = DRIVE_ROOT / 'NeuroMind_PRODUCTION_PREPARED_v15.zip'
assert PROJECT_ZIP.is_file(), (
    f'Project ZIP not found at {PROJECT_ZIP}. Upload the downloaded ZIP to the top level of My Drive.'
)

root_candidates = [
    p for p in DRIVE_ROOT.iterdir()
    if p.is_dir() and p.name.strip().lower() == 'neuromind' and (p / '01_Datasets').is_dir()
]
if len(root_candidates) != 1:
    print('Folders in My Drive named NeuroMind that contain 01_Datasets:')
    print([str(p) for p in root_candidates])
    raise RuntimeError(
        'Expected exactly one NeuroMind folder containing 01_Datasets. If your folder is elsewhere, edit DATA_ROOT manually.'
    )
DATA_ROOT = root_candidates[0]
print('Project ZIP:', PROJECT_ZIP)
print('Dataset root:', DATA_ROOT)
print('Drive folder name is shown exactly above; a trailing space is okay.')

In [ ]:
# Extract the project to Colab's local disk. This avoids running Python training directly on Drive.
EXTRACT_ROOT = Path('/content/neuromind_project')
if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
shutil.unpack_archive(str(PROJECT_ZIP), str(EXTRACT_ROOT))

project_candidates = [
    p for p in EXTRACT_ROOT.rglob('training')
    if p.is_dir() and (p / 'prepare_speech_embeddings.py').is_file()
]
assert len(project_candidates) == 1, f'Could not uniquely locate project training folder: {project_candidates}'
PROJECT_ROOT = project_candidates[0].parent
os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)
print('Project root exists:', PROJECT_ROOT.is_dir())

In [ ]:
# Install the training dependencies and verify GPU availability.
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'training/requirements-phase1.txt')],
    check=True,
)
import torch
import pandas as pd

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU not enabled. In Colab select Runtime → Change runtime type → T4 GPU, then rerun this cell.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Find the real data files. If your filenames differ, edit the paths in this cell explicitly.
def norm_name(value):
    return ' '.join(str(value).strip().lower().replace('_', ' ').replace('-', ' ').split())

# Find a speech folder that directly contains the two class folders.
non_names = {'non dementia', 'healthy', 'control', 'non demented'}
dementia_names = {'dementia', 'alzheimers', "alzheimer's", 'ad'}
audio_candidates = []
for p in DATA_ROOT.rglob('*'):
    if not p.is_dir() or not norm_name(p.name).startswith('speech dataset'):
        continue
    try:
        child_names = {norm_name(c.name) for c in p.iterdir() if c.is_dir()}
    except PermissionError:
        continue
    if child_names.intersection(non_names) and child_names.intersection(dementia_names):
        audio_candidates.append(p)
if len(audio_candidates) != 1:
    print('Speech folder candidates:', [str(p) for p in audio_candidates])
    raise RuntimeError('Expected exactly one Speech Dataset folder with Non-Dementia/Healthy and Dementia subfolders.')
AUDIO_ROOT = audio_candidates[0]

def find_csv(filename, folder_hint):
    matches = [p for p in DATA_ROOT.rglob('*.csv') if p.is_file() and p.name.strip().lower() == filename.lower()]
    preferred = [p for p in matches if any(norm_name(part) == norm_name(folder_hint) for part in p.parts)]
    if len(preferred) == 1:
        return preferred[0]
    if not preferred and len(matches) == 1:
        return matches[0]
    print(f'{filename} matches:', [str(p) for p in matches])
    raise RuntimeError(f'Could not uniquely locate {filename}. Edit its path manually in this cell.')

MEMORY_CSV = find_csv('Memory_Dataset.csv', 'Memory Dataset')
ATTENTION_CSV = find_csv('Attention_Dataset.csv', 'Attention Dataset')
LIFESTYLE_CSV = find_csv('Lifestyle_Dataset.csv', 'Lifestyle Dataset')

SEQUENCE_DIR = DATA_ROOT / '03_Features' / 'final_speech_sequences'
SPEECH_OUTPUT = DATA_ROOT / '04_Models' / 'Speech_Final'
FUSION_DATASET = DATA_ROOT / '03_Features' / 'final_fusion_dataset.csv'
FUSION_OUTPUT = DATA_ROOT / '04_Models' / 'Multimodal_Final'
SPEECH_LABEL_MAP = PROJECT_ROOT / 'training' / 'speech_label_map.example.json'
FUSION_LABEL_MAP = PROJECT_ROOT / 'training' / 'fusion_label_map.example.json'

for label, path in [
    ('speech', AUDIO_ROOT), ('memory', MEMORY_CSV),
    ('attention', ATTENTION_CSV), ('lifestyle', LIFESTYLE_CSV),
]:
    print(f'{label}: {path} | exists={path.exists()}')

# Basic inventory before starting expensive GPU work.
for class_folder in sorted(p for p in AUDIO_ROOT.iterdir() if p.is_dir()):
    patient_folders = [p for p in class_folder.iterdir() if p.is_dir()]
    audio_files = [p for p in class_folder.rglob('*') if p.is_file() and p.suffix.lower() in {'.wav','.mp3','.flac','.ogg','.m4a','.aac'}]
    print(f'{class_folder.name}: patient folders={len(patient_folders)}, audio files={len(audio_files)}')
for name, path in [('memory', MEMORY_CSV), ('attention', ATTENTION_CSV), ('lifestyle', LIFESTYLE_CSV)]:
    frame = pd.read_csv(path, keep_default_na=False)
    print(name, 'rows=', len(frame), 'columns=', list(frame.columns))

print('Check this inventory carefully. All modalities must use the same patient IDs and agree on the binary labels.')

In [ ]:
# Step 1: extract Wav2Vec2 sequence features for the real speech recordings.
# Resume mode reuses existing sequence files if they already exist.
def run_cmd(args):
    args = [str(x) for x in args]
    print('RUN:', ' '.join(args))
    subprocess.run(args, cwd=PROJECT_ROOT, check=True)

run_cmd([
    sys.executable, '-m', 'training.prepare_speech_embeddings',
    '--audio-root', AUDIO_ROOT,
    '--output-dir', SEQUENCE_DIR,
    '--label-map', SPEECH_LABEL_MAP,
    '--resume',
])

## Step 2 — train the speech encoder

This cell reports held-out patient metrics. The two `0.0` release cutoffs below allow exporting a research artifact for **technical integration testing** even if the metrics are weak. They do not mean the model is clinically validated. If your supervisor has specified release thresholds, enter those thresholds **before training** and do not change them after viewing the test results. Do not use this prototype for patient diagnosis.

In [ ]:
run_cmd([
    sys.executable, '-m', 'training.train_speech',
    '--manifest', SEQUENCE_DIR / 'speech_sequence_manifest.csv',
    '--output-dir', SPEECH_OUTPUT,
    '--epochs', '60',
    '--batch-size', '8',
    '--patience', '10',
    '--train-frame-mask', '0.03',
    '--min-test-macro-f1', '0.0',
    '--min-dementia-recall', '0.0',
])

speech_metrics_path = SPEECH_OUTPUT / 'speech_test_metrics.json'
print('Speech test metrics:')
print(speech_metrics_path.read_text() if speech_metrics_path.exists() else 'Metrics file not found')
print('Research prototype only: interpret metrics cautiously; this small dataset does not establish clinical performance.')

In [ ]:
# Step 3: build a patient-aligned multimodal dataset.
# This deliberately stops if patient IDs, labels, feature columns, or provenance do not match.
run_cmd([
    sys.executable, '-m', 'training.build_fusion_dataset',
    '--speech', SPEECH_OUTPUT / 'speech_patient_embeddings.csv',
    '--memory', MEMORY_CSV,
    '--attention', ATTENTION_CSV,
    '--lifestyle', LIFESTYLE_CSV,
    '--output', FUSION_DATASET,
    '--label-map', FUSION_LABEL_MAP,
])

fusion_frame = pd.read_csv(FUSION_DATASET)
print('Aligned patient count:', fusion_frame['patient_id'].nunique())
print(fusion_frame.groupby(['split', 'label']).size())
print('If this count is unexpectedly small, stop and fix patient IDs/labels; do not invent or synthesize missing patient data.')

In [ ]:
# Step 4: train the binary multimodal Transformer using the same speech model above.
run_cmd([
    sys.executable, '-m', 'training.train_fusion',
    '--dataset', FUSION_DATASET,
    '--speech-artifact', SPEECH_OUTPUT / 'neuromind_speech_bilstm_attention.pt',
    '--output-dir', FUSION_OUTPUT,
    '--epochs', '100',
    '--ablation-epochs', '50',
    '--batch-size', '16',
    '--patience', '15',
])
print('Fusion test metrics:')
print((FUSION_OUTPUT / 'fusion_test_metrics.json').read_text())

In [ ]:
# Step 5: validate TorchScript artifacts, manifests, class order, feature order, and speech/fusion lineage.
SPEECH_ARTIFACT = SPEECH_OUTPUT / 'neuromind_speech_bilstm_attention.pt'
FUSION_ARTIFACT = FUSION_OUTPUT / 'neuromind_multimodal_transformer_binary.pt'
run_cmd([
    sys.executable, '-m', 'training.validate_phase1_artifacts',
    '--speech-artifact', SPEECH_ARTIFACT,
    '--fusion-artifact', FUSION_ARTIFACT,
])

In [ ]:
# Step 6: install the exact trained artifacts and offline Wav2Vec2 backbone into the project.
# This is the step that connects the trained model files to the backend's expected ai_models/ paths.
run_cmd([
    sys.executable, '-m', 'training.install_phase1_artifacts',
    '--speech-artifact', SPEECH_ARTIFACT,
    '--fusion-artifact', FUSION_ARTIFACT,
    '--wav2vec-backbone', SEQUENCE_DIR / 'wav2vec2_backbone',
    '--project-root', PROJECT_ROOT,
])
print('Installed model files:')
for p in sorted((PROJECT_ROOT / 'ai_models').rglob('*')):
    if p.is_file():
        print(p.relative_to(PROJECT_ROOT), round(p.stat().st_size / (1024*1024), 2), 'MiB')

In [ ]:
# Step 7: package and checksum the deployable model bundle, then keep a copy in Drive.
run_cmd([sys.executable, 'scripts/package_model_bundle.py'])
BUNDLE = PROJECT_ROOT / 'dist' / 'neuromind-model-bundle.zip'
assert BUNDLE.is_file() and BUNDLE.stat().st_size > 0

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

DRIVE_BUNDLE = DATA_ROOT / '04_Models' / 'neuromind-model-bundle.zip'
DRIVE_BUNDLE.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(BUNDLE, DRIVE_BUNDLE)
print('MODEL BUNDLE CREATED')
print('Bundle saved in Drive:', DRIVE_BUNDLE)
print('Bundle size (MiB):', round(DRIVE_BUNDLE.stat().st_size / (1024*1024), 2))
print('MODEL_BUNDLE_SHA256=' + sha256_file(DRIVE_BUNDLE))
print('Keep the ZIP private. Do not commit the model bundle, datasets, or secrets to a public GitHub repository.')

## What you should have at the end

- `NeuroMind/04_Models/Speech_Final/neuromind_speech_bilstm_attention.pt` plus its manifest and test metrics.
- `NeuroMind/04_Models/Multimodal_Final/neuromind_multimodal_transformer_binary.pt` plus its manifest and test metrics.
- `NeuroMind/04_Models/neuromind-model-bundle.zip` — the deployable model bundle.
- The printed `MODEL_BUNDLE_SHA256` value — save this exact value for deployment configuration.

If a cell fails, stop and fix that error before continuing. Common blockers are missing dataset files, incompatible column names, patient IDs that do not match across modalities, label conflicts, too few patients in a train/validation/test split, or poor data quality. Do not manufacture rows or change labels just to make training pass.

The model bundle is **not yet hosted for production**. For local Docker testing, extract the bundle into the project's `ai_models/` directory so it creates `ai_models/speech/` and `ai_models/multimodal/`. For production Compose, upload the ZIP to stable private HTTPS artifact storage, then set `MODEL_BUNDLE_URL` and the printed `MODEL_BUNDLE_SHA256` in `.env.production.compose`. Never put the model URL/token in frontend `VITE_*` variables or commit secrets.